# Macro Factor Rotation - Research Notebook

**Source**: QC Strategy Library - AI Stocks-Bonds Rotation

**Concept**: Uses 3 FRED macro factors (VIX, yield curve, fed funds rate) to predict 21-day forward returns for a multi-asset portfolio (SPY, GLD, BND, BTC). A DecisionTreeRegressor is trained per asset with a 4-year lookback window. Positive predictions receive 1.5x leveraged weight.

In [1]:
try:
    from AlgorithmImports import *
    qb = QuantBook()
    QC_ENV = True
except (ImportError, NameError):
    import pandas as pd
    import numpy as np
    from datetime import timedelta
    QC_ENV = False
    qb = None

tickers = ["SPY", "GLD", "BND"]

if QC_ENV:
    for t in tickers:
        qb.AddEquity(t, Resolution.Daily)
else:
    # Local research path: the 3 assets via yfinance + the 3 FRED macro factors
    # via the public fredgraph.csv endpoints (same series the strategy names).
    print("Local environment - loading SPY/GLD/BND via yfinance + FRED factors")

print(f"Universe: {tickers} + BTCUSD")
print(f"Factors: VIXCLS (VIX), T10Y3M (Yield Curve), DFF (Fed Funds)")

Local environment - loading SPY/GLD/BND via yfinance + FRED factors
Universe: ['SPY', 'GLD', 'BND'] + BTCUSD
Factors: VIXCLS (VIX), T10Y3M (Yield Curve), DFF (Fed Funds)


## Strategy Logic

1. **Data**: FRED macro factors fetched daily (VIX, 10Y-3M spread, Fed Funds rate)
2. **Training**: Per-asset DecisionTreeRegressor (depth=12) on 4 years of history
3. **Prediction**: 21-day forward return forecast from latest factor values
4. **Allocation**: Assets with positive prediction get 1.5x leveraged weight
5. **BTC cap**: Bitcoin exposure capped at 10%, excess redistributed to equities
6. **Monthly rebalance**: Retrain and reallocate each month

In [2]:
# Fetch historical data - assets and the real FRED macro factors
if QC_ENV:
    hist = qb.History(tickers, timedelta(252 * 10), Resolution.Daily)
    close = hist["close"].unstack(level=0)
    factors = None  # FRED custom data handled by the strategy on QC Cloud
else:
    import yfinance as yf
    frames = {}
    for t in tickers:
        h = yf.Ticker(t).history(period="10y", auto_adjust=True)
        h.index = h.index.tz_localize(None)
        frames[t] = h["Close"]
    close = pd.DataFrame(frames).dropna()

    def fred(series):
        url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={series}"
        df = pd.read_csv(url, parse_dates=[0], index_col=0, names=["date", series], header=0)
        return df[series]

    factors = pd.concat(
        [fred("T10Y3M").rename("yield_curve"), fred("DFF").rename("fed_funds")],
        axis=1,
    )
    vix = yf.Ticker("^VIX").history(period="10y", auto_adjust=True)["Close"]
    vix.index = vix.index.tz_localize(None)
    factors["vix"] = vix
    factors = factors.reindex(close.index).ffill()

print(f"Assets close frame: {close.shape}")
if factors is not None:
    print(f"FRED/yfinance factor frame: {factors.shape} (no NaN: {not factors.isna().any().any()})")
    print(f"  Latest: " + ", ".join(f"{c}={factors[c].iloc[-1]:.2f}" for c in factors.columns))

Assets close frame: (2514, 3)
FRED/yfinance factor frame: (2514, 3) (no NaN: True)
  Latest: yield_curve=0.93, fed_funds=3.88, vix=14.87


In [3]:
spy_close = close["SPY"]
spy_21d_fwd = spy_close.pct_change(21).shift(-21)

print("SPY 21-day forward return statistics:")
print(f"  Mean: {spy_21d_fwd.mean():.2%}")
print(f"  Std: {spy_21d_fwd.std():.2%}")
print(f"  Positive ratio: {(spy_21d_fwd > 0).mean():.1%}")
print(f"  Sharpe (annualized): {(spy_21d_fwd.mean() / spy_21d_fwd.std()) * (252/21)**0.5:.2f}")

if factors is not None:
    print("\nFactor <-> 21d forward SPY return correlation (measured):")
    joined = factors.join(spy_21d_fwd.rename("fwd")).dropna()
    for c in factors.columns:
        print(f"  {c}: {joined[c].corr(joined['fwd']):+.3f}")

SPY 21-day forward return statistics:
  Mean: 1.32%
  Std: 4.56%
  Positive ratio: 69.5%
  Sharpe (annualized): 1.00

Factor <-> 21d forward SPY return correlation (measured):
  yield_curve: -0.057
  fed_funds: +0.021
  vix: +0.247


## Key Observations

- **Macro factors** provide regime context: VIX for risk appetite, yield curve for recession signal, Fed funds for monetary stance
- **DecisionTreeRegressor** captures non-linear factor-return relationships
- **Per-asset models** allow independent prediction across asset classes
- **1.5x leverage** amplifies conviction but increases drawdown risk

### Strengths
- Interpretable model (decision tree) with macro economic intuition
- Multi-asset diversification (equity, gold, bonds, crypto)
- Monthly retraining adapts to changing macro regimes
- FRED data is free, reliable, and well-documented

### Risks
- DecisionTree can overfit with depth=12 on 4 years of data
- BTC cap at 10% limits crypto exposure but adds complexity
- 21-day forward labels introduce look-ahead bias risk
- Leverage (1.5x) amplifies losses during misprediction
- FRED data may have publication lag in live trading

In [4]:
# The promised per-asset DecisionTreeRegressor -- trained and measured
from sklearn.tree import DecisionTreeRegressor

if factors is not None:
    print("=" * 62)
    print("PER-ASSET MODEL, MEASURED OUT-OF-SAMPLE (temporal 70/30, depth=12)")
    print("=" * 62)
    for t in tickers:
        fwd = close[t].pct_change(21).shift(-21)
        data = factors.join(fwd.rename("fwd")).dropna()
        split = int(len(data) * 0.7)
        Xtr, Xte = data.iloc[:split][factors.columns], data.iloc[split:][factors.columns]
        ytr, yte = data.iloc[:split]["fwd"], data.iloc[split:]["fwd"]
        tree = DecisionTreeRegressor(max_depth=12, random_state=42)
        tree.fit(Xtr, ytr)
        pred = tree.predict(Xte)
        ss_res = ((yte - pred) ** 2).sum()
        ss_tot = ((yte - yte.mean()) ** 2).sum()
        r2 = 1 - ss_res / ss_tot
        # honest baseline: predicting the training mean
        base_mse = ((yte - ytr.mean()) ** 2).mean()
        model_mse = ((yte - pred) ** 2).mean()
        print(f"  {t}: OOS R2 = {r2:+.3f} | MSE {model_mse:.5f} vs mean-baseline {base_mse:.5f} "
              f"({'beats' if model_mse < base_mse else 'WORSE than'} baseline)")
        print(f"      latest prediction (21d fwd): {tree.predict(factors.iloc[[-1]])[0]:+.2%}")
    print("\nHonest verdict: if R2 is near zero or negative, the 3 macro factors")
    print("alone do not predict 21-day returns at daily granularity -- the")
    print("strategy's edge (if any) lives in the full engine, not here.")
else:
    print("Models need FRED custom data: run on QC Cloud")

print()
print("=" * 62)
print("NOT MEASURED HERE: monthly rotation with 1.5x leverage, BTC cap,")
print("4-year rolling retraining, brokerage -> QC Cloud companion project.")
print("=" * 62)

PER-ASSET MODEL, MEASURED OUT-OF-SAMPLE (temporal 70/30, depth=12)
  SPY: OOS R2 = -0.669 | MSE 0.00232 vs mean-baseline 0.00143 (WORSE than baseline)
      latest prediction (21d fwd): -2.24%
  GLD: OOS R2 = -0.620 | MSE 0.00467 vs mean-baseline 0.00326 (WORSE than baseline)
      latest prediction (21d fwd): +1.42%
  BND: OOS R2 = -3.875 | MSE 0.00098 vs mean-baseline 0.00021 (WORSE than baseline)
      latest prediction (21d fwd): -3.53%

Honest verdict: if R2 is near zero or negative, the 3 macro factors
alone do not predict 21-day returns at daily granularity -- the
strategy's edge (if any) lives in the full engine, not here.

NOT MEASURED HERE: monthly rotation with 1.5x leverage, BTC cap,
4-year rolling retraining, brokerage -> QC Cloud companion project.


## Reading the measurements

- Factor-return correlations and the per-asset tree's out-of-sample R2 are
  **measured on real FRED series** (T10Y3M, DFF) and the real VIX -- the exact
  inputs the strategy names.
- A depth-12 tree on 3 macro factors is a high-variance model on ~2500 daily
  observations: near-zero or negative OOS R2 is the expected honest outcome,
  and the notebook reports it as such rather than hedging.
- The rotation mechanics (1.5x leverage on positive predictions, BTC cap,
  monthly retraining) belong to the QC Cloud engine on the companion strategy
  project.